# Learnofy Exploratory Analysis

This notebook analyzes synthetic sample data created from the existing Learnofy schema. It does not contain production user data.


In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path('../../data/sample')
students = pd.read_csv(DATA_DIR / 'synthetic_students.csv', parse_dates=['created_at'])
activity = pd.read_csv(DATA_DIR / 'synthetic_student_activity_daily.csv', parse_dates=['activity_date'])
sessions = pd.read_csv(DATA_DIR / 'synthetic_study_sessions.csv', parse_dates=['session_date'])
points = pd.read_csv(DATA_DIR / 'synthetic_points_transactions.csv', parse_dates=['timestamp'])
promotions = pd.read_csv(DATA_DIR / 'synthetic_promotions.csv', parse_dates=['start_at', 'end_at'])
student_promotions = pd.read_csv(DATA_DIR / 'synthetic_student_promotions.csv', parse_dates=['viewed_at', 'claimed_at', 'redeemed_at'])
feature_usage = pd.read_csv(DATA_DIR / 'synthetic_feature_usage.csv', parse_dates=['event_date'])


In [ ]:
activity['week'] = activity['activity_date'].dt.to_period('W').dt.start_time
weekly_engagement = activity.groupby('week').agg(
    weekly_active_users=('student_id', lambda s: s[activity.loc[s.index, 'active_seconds'] > 0].nunique()),
    study_hours=('active_seconds', lambda s: s.sum() / 3600),
    active_days=('active_seconds', lambda s: (s > 0).sum())
).reset_index()
weekly_engagement


In [ ]:
student_summary = activity.groupby('student_id').agg(
    active_days=('active_seconds', lambda s: (s > 0).sum()),
    total_study_hours=('active_seconds', lambda s: s.sum() / 3600),
    avg_daily_minutes=('active_seconds', lambda s: s.mean() / 60)
).merge(students[['student_id', 'segment_label', 'subscription_status']], on='student_id')
student_summary.groupby('segment_label')[['active_days', 'total_study_hours', 'avg_daily_minutes']].mean().round(2)


In [ ]:
promo_performance = student_promotions.merge(promotions[['promotion_id', 'cafe_name', 'title']], on='promotion_id')
promo_performance['claimed'] = promo_performance['claimed_at'].notna()
promo_performance['redeemed'] = promo_performance['redeemed_at'].notna()
promo_summary = promo_performance.groupby('cafe_name').agg(
    views=('student_promotion_id', 'count'),
    claims=('claimed', 'sum'),
    redemptions=('redeemed', 'sum')
)
promo_summary['claim_rate'] = promo_summary['claims'] / promo_summary['views']
promo_summary.sort_values('claim_rate', ascending=False).round(3)


In [ ]:
feature_usage.groupby(['feature_name', 'event_type']).agg(events=('event_id', 'count'), users=('student_id', 'nunique')).sort_values('events', ascending=False)


## Chart Outputs

The repository includes exported PNG charts in `python/outputs/` so recruiters can review the visual results without running the notebook.
